# 3.8 Agregasyon ve Gruplama

Bu notebook, PDS Handbook (TR) web sayfasının **Türkçe Jupyter karşılığıdır** — aynı açıklamalar, ders notları ve kod örnekleri.

| | |
|---|---|
| **Web sayfası** | `chapters/03-pandas/08-aggregation-and-grouping.html` |
| **Çalıştırma** | JupyterLab, VS Code veya Colab — hücreleri **yukarıdan aşağı** sırayla (`Shift+Enter`) |
| **Bağımlılık** | Kod hücreleri birbirine bağlıdır; hata alırsanız önce üsttekileri çalıştırın |

> **Kaynak:** Jake VanderPlas, *Python Data Science Handbook* — Türkçe ders uyarlaması



Orijinal: Aggregation and Grouping

Birçok veri analizi görevinin temel parçası verimli özetlemedir: büyük bir veri kümesinin belirli yönlerini tek sayıyla özetleyen sum, mean, median, min, max gibi agregasyonlar. Bu bölümde Pandas'ta NumPy dizilerindekine benzer basit işlemlerden groupby kavramına dayalı daha gelişmiş işlemlere geçeceğiz.

Kolaylık için önceki bölümlerdeki display yardımcı sınıfını kullanacağız:


In [ ]:
# imports_display.py
import numpy as np
import pandas as pd

class display(object):
    """Display HTML representation of multiple objects"""
    template = """<div style="float: left; padding: 10px;">
    <p style='font-family:"Courier New", Courier, monospace'>{0}</p>{1}
    </div>"""
    def __init__(self, *args):
        self.args = args
        
    def _repr_html_(self):
        return '\n'.join(self.template.format(a, eval(a)._repr_html_())
                         for a in self.args)
    
    def __repr__(self):
        return '\n\n'.join(a + '\n' + repr(eval(a))
                           for a in self.args)



## Planets Verisi

Seaborn paketindeki Planets veri kümesini kullanacağız (bkz. görselleştirme bölümleri). Diğer yıldızların çevresinde keşfedilen ötegezegenlere ilişkin bilgi içerir; Seaborn ile indirilebilir:


In [ ]:
# load_planets.py
import seaborn as sns
planets = sns.load_dataset('planets')
planets.shape



In [ ]:
# planets_head.py
planets.head()



> **Not**
>

2014'e kadar keşfedilen 1000'den fazla ötegezegene ilişkin ayrıntılar içerir.

## Pandas'ta Basit Agregasyon

2.4 Agregasyonlar'da NumPy dizileri için agregasyonları gördük. Tek boyutlu Series için agregatlar tek değer döndürür:


In [ ]:
# ser_random.py
rng = np.random.RandomState(42)
ser = pd.Series(rng.rand(5))
ser



In [ ]:
# ser_sum.py
ser.sum()



In [ ]:
# ser_mean.py
ser.mean()



DataFrame için varsayılan olarak agregatlar her sütun içinde sonuç üretir:


In [ ]:
# df_random.py
df = pd.DataFrame({'A': rng.rand(5),
                   'B': rng.rand(5)})
df



In [ ]:
# df_mean.py
df.mean()



axis ile satırlar boyunca da agregasyon yapılabilir:


In [ ]:
# df_mean_axis_columns.py
df.mean(axis='columns')



Pandas Series ve DataFrame ortak agregasyonların yanı sıra her sütun için birkaç özet veren describe yöntemini içerir. Eksik satırları düşürerek Planets verisinde deneyelim:


In [ ]:
# planets_describe.py
planets.dropna().describe()



Bu yöntem veri kümesinin genel özelliklerini anlamaya yardımcı olur. Örneğin year sütununda ötegezegenler 1989'dan beri keşfedilmiş olsa da veri kümesindeki gezegenlerin yarısı 2010 veya sonrasında keşfedilmiş — büyük ölçüde Kepler görevi sayesinde.

Pandas'taki diğer yerleşik agregasyonların özeti:

Bunların hepsi DataFrame ve Series yöntemleridir.

Daha derine inmek için basit agregatlar çoğu zaman yeterli değildir. Bir sonraki düzey groupby işlemidir — verinin alt kümeleri üzerinde hızlı ve verimli agregasyon.

## groupby: Böl, Uygula, Birleştir

Basit agregatlar veri kümesinin tadını verir; çoğu zaman etiket veya indekse göre koşullu agregasyon isteriz — SQL'deki “group by” komutundan gelen ad; Hadley Wickham'ın ifadesiyle split, apply, combine (böl, uygula, birleştir).

### Split, Apply, Combine

“Uygula” adımının toplama agregasyonu olduğu kanonik örnek aşağıdaki şekilde gösterilir:

groupby şunları yapar:

Bunu maskeleme, agregasyon ve birleştirme ile elle yapabilirdiniz; önemli nokta: ara bölümlerin açıkça oluşturulması gerekmez. groupby çoğu zaman tek geçişte her grup için toplam, ortalama, sayım vb. günceller. Gücü, kullanıcının alttaki hesabı düşünmeden işlemi bir bütün olarak görmesidir.

Somut örnek için girdi DataFrame'ini oluşturalım:


In [ ]:
# groupby_df_example.py
df = pd.DataFrame({'key': ['A', 'B', 'C', 'A', 'B', 'C'],
                   'data': range(6)}, columns=['key', 'data'])
df



En temel split-apply-combine, DataFrame'in groupby yöntemiyle istenen anahtar sütun adının verilmesiyle hesaplanır:


In [ ]:
# df_groupby_key.py
df.groupby('key')



Dönen nesne bir DataFrameGroupBy nesnesidir, DataFrame kümesi değil. Gruplara “hazır” özel bir görünüm düşünün; agregasyon uygulanana kadar hesap yapılmaz (tembel değerlendirme).


In [ ]:
# df_groupby_sum.py
df.groupby('key').sum()



sum yalnızca bir seçenektir; çoğu Pandas/NumPy agregasyonu ve birçok DataFrame işlemi uygulanabilir.

### GroupBy Nesnesi

GroupBy esnek bir soyutlamadır; altta daha gelişmiş işlemler yapılır. Planets verisiyle örnekler: aggregate, filter, transform, apply — sonraki alt bölümde; önce temel GroupBy işlevleri.

#### Sütun indeksleme

GroupBy, DataFrame gibi sütun indekslemesini destekler:


In [ ]:
# planets_groupby_method.py
planets.groupby('method')



In [ ]:
# planets_groupby_orbital.py
planets.groupby('method')['orbital_period']



Orijinal gruptan belirli bir Series seçildi; agregasyon çağrılana kadar hesap yok:


In [ ]:
# planets_orbital_median.py
planets.groupby('method')['orbital_period'].median()



Her yöntemin duyarlı olduğu yörünge periyodu (gün) ölçeğine dair fikir verir.

#### Gruplar üzerinde döngü

GroupBy gruplar üzerinde doğrudan yineleme destekler:


In [ ]:
# planets_groupby_iter.py
for (method, group) in planets.groupby('method'):
    print("{0:30s} shape={1}".format(method, group.shape))



Hata ayıklma için elle incelemede yararlıdır; çoğu zaman yerleşik apply daha hızlıdır.

#### Dispatch yöntemleri

GroupBy'da açıkça tanımlanmayan yöntemler gruplara iletilir; describe her grup için describe çağırmaya eşdeğerdir:


In [ ]:
# planets_year_describe.py
planets.groupby('method')['year'].describe().unstack()



Bu tablo veriyi anlamaya yardımcı olur: 2014'e kadar gezegenlerin büyük çoğunluğu Radial Velocity ve Transit yöntemleriyle keşfedilmiş; Transit son yıllarda yaygınlaşmış. Transit Timing Variation ve Orbital Brightness Modulation 2011'den sonra kullanılmış.

### Aggregate, Filter, Transform, Apply

Önceki tartışma birleştirme için agregasyona odaklandı; GroupBy'da aggregate, filter, transform ve apply de vardır. Aşağıdaki alt bölümler için örnek DataFrame:


In [ ]:
# agg_filter_df.py
rng = np.random.RandomState(0)
df = pd.DataFrame({'key': ['A', 'B', 'C', 'A', 'B', 'C'],
                   'data1': range(6),
                   'data2': rng.randint(0, 10, 6)},
                   columns = ['key', 'data1', 'data2'])
df



#### Aggregation

sum, median ile tanıdık agregasyonların ötesinde aggregate daha fazla esneklik sunar — dize, fonksiyon veya liste alıp hepsini birden hesaplayabilir:


In [ ]:
# groupby_aggregate_multi.py
df.groupby('key').aggregate(['min', np.median, max])



Yaygın desen: sütun adlarını o sütuna uygulanacak işlemlere eşleyen sözlük:


In [ ]:
# groupby_aggregate_dict.py
df.groupby('key').aggregate({'data1': 'min',
                             'data2': 'max'})



#### Filtering

Filtreleme, grup özelliklerine göre veriyi düşürür. Örneğin standart sapması eşiğin üstündeki grupları tutmak:


In [ ]:
# groupby_filter_func.py
def filter_func(x):
    return x['data2'].std() > 4

display('df', "df.groupby('key').std()",
        "df.groupby('key').filter(filter_func)")



Filtre fonksiyonu grubun geçip geçmeyeceğini belirten Boolean döndürmelidir. Burada A grubunun standart sapması 4'ten büyük olmadığı için düşürülür.

#### Transformation

Agregasyon verinin küçültülmüş biçimini döndürürken dönüşüm, birleştirmek için tam verinin dönüştürülmüş biçimini döndürür — çıktı girişle aynı şekildedir. Yaygın örnek: grup ortalamasını çıkararak ortalamak:


In [ ]:
# groupby_transform_center.py
def center(x):
    return x - x.mean()
df.groupby('key').transform(center)



#### apply yöntemi

apply, gruba keyfi bir fonksiyon uygular. Fonksiyon DataFrame alıp Pandas nesnesi veya skaler döndürmelidir. Örneğin ilk sütunu ikincinin toplamına göre normalize etmek:


In [ ]:
# groupby_apply_norm.py
def norm_by_data2(x):
    # x is a DataFrame of group values
    x['data1'] /= x['data2'].sum()
    return x

df.groupby('key').apply(norm_by_data2)



> **Not**
>

GroupBy içinde apply esnektir; tek ölçüt fonksiyonun DataFrame alıp Pandas nesnesi veya skaler döndürmesidir.

### Bölme Anahtarının Belirtilmesi

Basit örneklerde DataFrame tek sütun adına göre bölündü. Gruplar başka yollarla da tanımlanabilir.

#### Liste, dizi, seri veya indeks

Anahtar, DataFrame uzunluğuyla eşleşen herhangi bir seri veya liste olabilir:


In [ ]:
# groupby_list_L.py
L = [0, 1, 0, 1, 2, 0]
df.groupby(L).sum()



Bu, df.groupby('key')'nin daha ayrıntılı eşdeğeridir:


In [ ]:
# groupby_df_key.py
df.groupby(df['key']).sum()



#### Sözlük veya seri ile indeks eşlemesi

İndeks değerlerini grup anahtarlarına eşleyen sözlük de verilebilir:


In [ ]:
# groupby_mapping.py
df2 = df.set_index('key')
mapping = {'A': 'vowel', 'B': 'consonant', 'C': 'consonant'}
display('df2', 'df2.groupby(mapping).sum()')



#### Herhangi bir Python fonksiyonu

İndeks değerini alıp grup döndüren herhangi bir fonksiyon:


In [ ]:
# groupby_str_lower.py
df2.groupby(str.lower).mean()



#### Geçerli anahtarların listesi

Önceki anahtar seçenekleri çoklu indeks için birleştirilebilir:


In [ ]:
# groupby_multi_keys.py
df2.groupby([str.lower, mapping]).mean()



### Gruplama Örneği

Birkaç satır Python ile yöntem ve on yıla göre keşfedilen gezegen sayısını sayabiliriz:


In [ ]:
# planets_decade_method.py
decade = 10 * (planets['year'] // 10)
decade = decade.astype(str) + 's'
decade.name = 'decade'
planets.groupby(['method', decade])['number'].sum().unstack().fillna(0)



Bu, gerçekçi veri kümelerinde öğrendiğimiz işlemleri birleştirmenin gücünü gösterir: ötegezegenlerin ilk keşfinden sonra ne zaman ve nasıl tespit edildiğine dair kaba bir resim hızla elde edilir.

Bu kod satırlarını adım adım inceleyip her adımın sonuca ne yaptığını anlamanızı öneririm — karmaşık görünse de parçaları anlamak kendi verinizi keşfetmenin yolunu açar.

### 🧪 Şimdi deneyin

🧪 Şimdi deneyin
      planets verisinde (veya küçük bir örnek çerçevede) groupby('method')['distance'].median() hesaplayın:
          
      import pandas as pd
try:
    import seaborn as sns
    planets = sns.load_dataset('planets')
    print(planets.groupby('method')['distance'].median())
except Exception as e:
    df = pd.DataFrame({'method': ['A','A','B'], 'distance': [1.0, 2.0, 5.0]})
    print(df.groupby('method')['distance'].median())

> **Not**
>
